# FWI projections 2026–2045

**SECS — Southern Europe Climate Study** · Seville (Guadalquivir basin) · Larissa (Thessaly plain)

Extends the fire-weather arm forward from the CEMS historical record (1990–2025) into the CMIP6
projection window (2026–2045) under two scenarios: **SSP2-4.5** (central) and **SSP5-8.5** (high).

---

## Why FWI is computed here rather than downloaded

There is no pre-computed FWI projection product that is compatible with this study's CMIP6 run.
The fire-danger projections available on CDS derive from EURO-CORDEX regional models under RCP
forcing — a different model, a different downscaling chain, a different scenario framing. Bolting
those onto an EC-Earth3-CC heat arm would mean the fire slide and the heat slide describe two
different simulated futures.

The only internally consistent option is to compute FWI from the same daily fields that drive the
heat indicators. That is what this notebook does.

## The wind substitution

EC-Earth3-CC does not publish `near_surface_wind_speed` at daily resolution under either SSP
scenario on CDS (confirmed against the CDS constraints endpoint — see `scenario_scan.py`). Wind is
therefore **held fixed at the ERA5 1991–2020 monthly climatology**.

This is defensible rather than merely convenient: near-surface wind is among the least reliable
CMIP6 outputs, and holding it fixed isolates the thermodynamic component of fire-weather change —
temperature, humidity and precipitation — which is the component that can actually be defended.

> **Caveat for the slide:** *wind held at historical climatology; projected FWI change reflects
> temperature, humidity and precipitation only.*

## What is actually being projected

The deliverables are `days_fwi_gt30` and `longest_high_run` — both threshold crossings at the
EFFIS "high" fire-danger class. Reproducing **exceedance frequency and persistence** is a lower
bar than reproducing FWI magnitude, and it is the bar this method clears. Absolute FWI values
carry a calibration offset (quantified in §3) and should not be quoted as such.

## Inputs

| Source | File | Role |
|---|---|---|
| CEMS (EWDS) | `data/seville_fwi_daily_1990_2025.csv` | historical FWI, validation target, warm start |
| ERA5 | `data/seville_era5_daily.csv` | wind climatology; validation run inputs |
| CMIP6 | `data/{city}_cmip6_{tag}_daily_2026_2045.csv` | projection inputs |

Computation lives in `fwi_from_daily.py` (smoke-tested standalone against synthetic data before
being used here). This notebook imports it rather than duplicating the logic.

## 0. Setup

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)

DATA = "data"
FIGS = "figures"
os.makedirs(FIGS, exist_ok=True)

# --- site configuration ------------------------------------------------
SITES = {
    "seville": {"lat": 37.39, "colour": "#E8A33D", "label": "Seville"},
    "larissa": {"lat": 39.64, "colour": "#3D9E8F", "label": "Larissa"},
}

SCENARIOS = {
    "ssp245": {"label": "SSP2-4.5", "colour": "#4A7BA7", "style": "--"},
    "ssp585": {"label": "SSP5-8.5", "colour": "#C4562A", "style": "-"},
}

FWI_THRESHOLD = 30.0     # EFFIS "high" fire danger
SPINUP_DAYS = 365        # discarded after integration

# ERA5 column names, from seville_era5_daily.csv
ERA5_MAP = {"tasmax": "t2m_max", "pr": "tp_mm"}
ERA5_RH_COL = "rh_min"
ERA5_WIND_COL = "wind_avg"

print("pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# Import the validated computation module. Fails loudly if it is not on the path.
sys.path.insert(0, os.getcwd())
import fwi_from_daily as F

import xclim
print("xclim", xclim.__version__)
print("module loaded:", F.__file__)

In [ ]:
def show(path):
    "Report whether an input file exists, and its shape if so."
    if os.path.exists(path):
        d = pd.read_csv(path, nrows=1)
        n = sum(1 for _ in open(path, encoding="utf-8")) - 1
        print(f"  OK      {path}  ({n} rows)")
        print(f"          {list(d.columns)}")
        return True
    print(f"  MISSING {path}")
    return False

print("Historical inputs")
have_era5 = show(f"{DATA}/seville_era5_daily.csv")
have_cems = show(f"{DATA}/seville_fwi_daily_1990_2025.csv")

print("\nProjection inputs")
proj_status = {}
for city in SITES:
    for tag in SCENARIOS:
        p = f"{DATA}/{city}_cmip6_{tag}_daily_2026_2045.csv"
        proj_status[(city, tag)] = show(p)

## 1. Wind climatology

Built from the ERA5 daily record already held — no additional download required.

`wind_avg` is a **daily mean**, not a noon value, and noon wind typically runs higher. The
climatology is therefore biased slightly low, which makes projected FWI conservative rather than
inflated. Noted, not corrected.

The monthly means are pinned to the 15th of each month and interpolated to daily inside
`fwi_from_daily.py`, wrapping across the year boundary. Applying them as step functions would put
a discontinuity at every month edge, which propagates into FFMC as visible sawtooth.

In [ ]:
CLIM_PATH = f"{DATA}/wind_climatology_noon.csv"

for city in SITES:
    src = f"{DATA}/{city}_era5_daily.csv"
    if not os.path.exists(src):
        print(f"[skip] {city}: no ERA5 daily file")
        continue
    F.build_clim_from_daily(
        daily_csv=src,
        wind_col=ERA5_WIND_COL,
        city=city,
        out_csv=CLIM_PATH,
        wind_units="m/s",
        start=1991, end=2020,
    )
    print()

In [ ]:
clim = pd.read_csv(CLIM_PATH)
display(clim.pivot(index="month", columns="city", values="wind_kmh").round(2))

fig, ax = plt.subplots(figsize=(8, 3.4))
for city, g in clim.groupby("city"):
    ax.plot(g["month"], g["wind_kmh"], marker="o", lw=2,
            color=SITES.get(city, {}).get("colour", "#888"),
            label=SITES.get(city, {}).get("label", city))
ax.set_xticks(range(1, 13))
ax.set_xlabel("month"); ax.set_ylabel("wind speed (km/h)")
ax.set_title("ERA5 1991–2020 monthly wind climatology (daily mean, 10 m)")
ax.grid(alpha=.3); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(f"{FIGS}/wind_climatology.png", dpi=150); plt.show()

## 2. Validation run — FWI on ERA5 historical inputs

This is the gate. Everything downstream is only worth running if the implementation reproduces the
CEMS series.

Two differences from the projection run, both deliberate:

- **Real wind is used** (`--wind-col wind_avg`), not the climatology. The climatology is a stand-in
  only where the model supplies nothing.
- **`rh_min` is used directly** as the noon humidity proxy. Evaluating RH at the daily maximum
  temperature yields the daily *minimum* RH, which is what `rh_min` already is. The CMIP6 side
  derives the same quantity from `huss` + `psl` + `tasmax`, so this validation tests the FWI code
  but **not** the RH derivation — an asymmetry worth stating rather than glossing.

In [ ]:
VALID_OUT = f"{DATA}/seville_fwi_era5_validation.csv"

work = F.prepare(
    daily_csv=f"{DATA}/seville_era5_daily.csv",
    wind_csv=CLIM_PATH,
    city="seville",
    colmap={**F.DEFAULT_MAP, **ERA5_MAP},
    rh_col=ERA5_RH_COL,
    wind_col=ERA5_WIND_COL,
    wind_units="m/s",
)

valid = F.compute_fwi(work, lat=SITES["seville"]["lat"])
valid = valid.iloc[SPINUP_DAYS:]
valid.to_csv(VALID_OUT, float_format="%.4f")
print(f"\nwritten {VALID_OUT}  ({len(valid)} rows)")
valid[["tasmax_c", "hurs_noon", "wind_kmh", "pr_mm", "ffmc", "dc", "fwi"]].tail(5).round(2)

### 2.1 Against CEMS

**Expected:** a systematic offset. The CEMS product is bias-corrected against reanalysis FWI over
1981–2010 and is not derived from ERA5 directly, so its absolute level sits on a different
calibration. That offset is a number to report, not a defect to fix.

**Pass criteria:**

| Code | Threshold | Why |
|---|---|---|
| `dc` (drought code) | r ≳ 0.95 | smooth, slow, months of memory — should track very tightly |
| `fwi` | r ≳ 0.85 | daily, noisier, sensitive to the wind and noon-timing mismatch |
| `ffmc` | r ≳ 0.75 | fastest-responding code, most sensitive to sub-daily timing |

A low `dc` correlation points at a unit or accumulation error, not a calibration difference — that
would be a genuine failure and a reason to stop.

In [ ]:
cems = pd.read_csv(f"{DATA}/seville_fwi_daily_1990_2025.csv",
                   index_col="date", parse_dates=True)

pairs = [("fwi", "fire_weather_index"),
         ("dc", "drought_code"),
         ("ffmc", "fine_fuel_moisture_code"),
         ("bui", "build_up_index"),
         ("isi", "initial_fire_spread_index")]

j = valid[[a for a, _ in pairs]].join(
        cems[[b for _, b in pairs]], how="inner").dropna()
print(f"overlapping days: {len(j)}  "
      f"({j.index.min().date()} -> {j.index.max().date()})\n")

rows = []
for a, b in pairs:
    rows.append({
        "code": a,
        "r": round(j[a].corr(j[b]), 3),
        "mine_mean": round(j[a].mean(), 2),
        "cems_mean": round(j[b].mean(), 2),
        "bias": round(j[a].mean() - j[b].mean(), 2),
    })
validation_table = pd.DataFrame(rows).set_index("code")
display(validation_table)

FWI_BIAS = float(validation_table.loc["fwi", "bias"])
FWI_R = float(validation_table.loc["fwi", "r"])
print(f"\n>>> quote on the slide: r = {FWI_R:.2f}, mean offset = {FWI_BIAS:+.2f} FWI units")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

ax = axes[0]
ax.scatter(j["fire_weather_index"], j["fwi"], s=3, alpha=.15, color="#C4562A")
lim = [0, max(j["fwi"].max(), j["fire_weather_index"].max())]
ax.plot(lim, lim, color="#1F3357", lw=1, ls="--", label="1:1")
ax.set_xlabel("CEMS FWI"); ax.set_ylabel("computed FWI")
ax.set_title(f"Daily FWI, r = {FWI_R:.3f}")
ax.legend(frameon=False); ax.grid(alpha=.3)

ax = axes[1]
yr = 2015
s = j.loc[str(yr)]
ax.plot(s.index, s["fire_weather_index"], lw=1.2, color="#1F3357", label="CEMS")
ax.plot(s.index, s["fwi"], lw=1.2, color="#C4562A", label="computed")
ax.axhline(FWI_THRESHOLD, color="#888", ls=":", lw=1, label=f"EFFIS high ({FWI_THRESHOLD:.0f})")
ax.set_title(f"Seasonal cycle, {yr}"); ax.set_ylabel("FWI")
ax.legend(frameon=False); ax.grid(alpha=.3)

plt.tight_layout(); plt.savefig(f"{FIGS}/fwi_validation.png", dpi=150); plt.show()

### 2.2 Annual indicators — do the metrics that matter agree?

Correlation on daily values is reassuring, but the study reports **annual** counts. This checks the
two quantities that actually reach the slide.

In [ ]:
def annual_from_series(fwi_series, threshold=FWI_THRESHOLD):
    df = pd.DataFrame({"fwi": fwi_series}).dropna()
    return F.annual_indicators(df, threshold)

ann_mine = annual_from_series(j["fwi"])
ann_cems = annual_from_series(j["fire_weather_index"])

cmp_ann = ann_mine[["days_fwi_gt30", "longest_high_run"]].join(
    ann_cems[["days_fwi_gt30", "longest_high_run"]],
    lsuffix="_mine", rsuffix="_cems")
display(cmp_ann)

for m in ["days_fwi_gt30", "longest_high_run"]:
    r = cmp_ann[f"{m}_mine"].corr(cmp_ann[f"{m}_cems"])
    b = cmp_ann[f"{m}_mine"].mean() - cmp_ann[f"{m}_cems"].mean()
    print(f"{m:18s} r = {r:.3f}   bias = {b:+.1f} days")

## 3. Warm start from CEMS

FWI is sequential and the drought code carries months of memory, so a cold start contaminates the
first season. Two options:

1. **Cold start + discard** — begin a full year early and throw that year away (`SPINUP_DAYS`).
2. **Warm start** — initialise from the CEMS codes at the end of 2025, removing the discontinuity
   at the historical/projection seam.

The warm start is preferred where available. Its one flaw: the CEMS codes carry the bias correction
noted in §2.1, so they sit on a slightly shifted scale. A shifted warm start still beats an
arbitrary cold one.

CEMS supplies DC, FFMC and BUI but **not** DMC directly. DMC is recoverable from BUI and DC by
inverting the BUI relation; where that inversion is ill-conditioned the xclim default is used
instead, and the spin-up discard covers the residual error.

In [ ]:
def warm_start_codes(cems_df, on="2025-12-31"):
    "Return (ffmc0, dmc0, dc0) from the CEMS record, or (None, None, None)."
    try:
        row = cems_df.loc[on]
    except KeyError:
        print(f"[warm start] {on} not in CEMS record -> cold start")
        return None, None, None

    dc0 = float(row["drought_code"])
    ffmc0 = float(row["fine_fuel_moisture_code"])
    bui = float(row["build_up_index"])

    # Invert BUI for DMC. BUI = 0.8*DMC*DC/(DMC+0.4*DC) when DMC <= 0.4*DC.
    dmc0 = None
    denom = 0.8 * dc0 - bui
    if denom > 1e-6:
        cand = 0.4 * dc0 * bui / denom
        if 0 < cand < 500:
            dmc0 = float(cand)
    if dmc0 is None:
        print("[warm start] DMC inversion ill-conditioned -> xclim default for DMC")

    print(f"[warm start] {on}: ffmc0={ffmc0:.2f} dmc0="
          f"{'%.2f' % dmc0 if dmc0 else 'default'} dc0={dc0:.2f}")
    return ffmc0, dmc0, dc0

FFMC0, DMC0, DC0 = warm_start_codes(cems)

## 4. Projection runs

One run per site per scenario. Cells skip gracefully where the CMIP6 download has not completed,
so the notebook stays runnable while SSP2-4.5 is still in the queue.

**Column names in the CMIP6 CSVs are probed rather than assumed** — the merge stage may emit
`tasmax`/`hurs_derived` or the raw CMIP6 names depending on how it was configured.

In [ ]:
def probe_map(path):
    "Guess the column mapping for a CMIP6 daily CSV."
    cols = list(pd.read_csv(path, nrows=1).columns)
    def pick(*cands):
        for cand in cands:
            for col in cols:
                if col.lower() == cand:
                    return col
        for cand in cands:
            for col in cols:
                if cand in col.lower():
                    return col
        return None

    m = {
        "tasmax": pick("tasmax", "t2m_max", "tx"),
        "pr": pick("pr_mm", "pr", "tp_mm", "precip"),
        "huss": pick("huss", "specific_humidity"),
        "psl": pick("psl", "sea_level_pressure"),
    }
    rh = pick("hurs_derived", "hurs", "rh_min", "rh")
    print(f"  columns : {cols}")
    print(f"  mapping : {m}")
    print(f"  rh col  : {rh}")
    return m, rh

In [ ]:
projections = {}

for city, cfg in SITES.items():
    for tag in SCENARIOS:
        src = f"{DATA}/{city}_cmip6_{tag}_daily_2026_2045.csv"
        if not os.path.exists(src):
            print(f"[skip] {city} {tag}: {src} not found")
            continue

        print(f"\n=== {cfg['label']} / {SCENARIOS[tag]['label']} ===")
        m, rh = probe_map(src)

        w = F.prepare(
            daily_csv=src, wind_csv=CLIM_PATH, city=city,
            colmap={**F.DEFAULT_MAP, **{k: v for k, v in m.items() if v}},
            rh_col=rh if rh and rh.startswith(("hurs", "rh")) else None,
            wind_col=None,           # climatology: the Option B substitution
        )
        out = F.compute_fwi(w, lat=cfg["lat"],
                            ffmc0=FFMC0, dmc0=DMC0, dc0=DC0)
        out = out.iloc[SPINUP_DAYS:] if FFMC0 is None else out

        dst = f"{DATA}/{city}_fwi_{tag}_2026_2045.csv"
        out.to_csv(dst, float_format="%.4f")
        projections[(city, tag)] = out
        print(f"  -> {dst}  ({len(out)} rows, "
              f"{out.index.min().date()} to {out.index.max().date()})")

print(f"\n{len(projections)} projection run(s) complete")

## 5. Annual indicators and trends

In [ ]:
def theil_sen(y, x=None):
    "Theil-Sen slope per year plus Kendall tau. Matches the historical arm."
    y = pd.Series(y).dropna()
    x = np.asarray(y.index, dtype=float) if x is None else np.asarray(x, dtype=float)
    if len(y) < 5:
        return dict(slope=np.nan, tau=np.nan, p=np.nan, n=len(y))
    slope, intercept, lo, hi = stats.theilslopes(y.values, x, 0.95)
    tau, p = stats.kendalltau(x, y.values)
    return dict(slope=slope, lo=lo, hi=hi, tau=tau, p=p, n=len(y))


rows = []
annuals = {}
for (city, tag), df in projections.items():
    ann = F.annual_indicators(df, FWI_THRESHOLD)
    annuals[(city, tag)] = ann
    ann.to_csv(f"{DATA}/{city}_fwi_{tag}_2026_2045_annual.csv")
    for metric in ["days_fwi_gt30", "longest_high_run", "fwi_mean", "fwi_p95"]:
        t = theil_sen(ann[metric])
        rows.append({
            "city": SITES[city]["label"],
            "scenario": SCENARIOS[tag]["label"],
            "metric": metric,
            "mean": round(ann[metric].mean(), 2),
            "slope_per_decade": round(t["slope"] * 10, 3),
            "kendall_tau": round(t["tau"], 3),
            "p_value": round(t["p"], 4),
            "n_years": t["n"],
        })

trend_table = pd.DataFrame(rows)
display(trend_table)

**Reading the trend table.** Over a 20-year window the sample is small and p-values will often sit
above 0.05 even where the direction is consistent. Report direction and magnitude; do not lean on
significance. This mirrors the finding from the baseline-model work — trend is detectable, but the
naive baseline wins on forecast skill, and neither result invalidates the other.

In [ ]:
if annuals:
    metrics = ["days_fwi_gt30", "longest_high_run"]
    fig, axes = plt.subplots(len(metrics), 1, figsize=(10, 4.2 * len(metrics)),
                             sharex=True)
    axes = np.atleast_1d(axes)

    for ax, metric in zip(axes, metrics):
        for (city, tag), ann in annuals.items():
            ax.plot(ann.index, ann[metric],
                    color=SCENARIOS[tag]["colour"],
                    ls=SCENARIOS[tag]["style"], lw=1.8, marker="o", ms=3,
                    label=f"{SITES[city]['label']} {SCENARIOS[tag]['label']}")
            t = theil_sen(ann[metric])
            x = np.asarray(ann.index, dtype=float)
            ax.plot(ann.index, t["slope"] * (x - x[0]) + ann[metric].iloc[0],
                    color=SCENARIOS[tag]["colour"], lw=1, alpha=.45)
        ax.set_ylabel(metric.replace("_", " "))
        ax.grid(alpha=.3)
        ax.legend(frameon=False, fontsize=8, ncol=2)

    axes[0].set_title("Projected fire-danger indicators, 2026–2045\n"
                      "(wind held at ERA5 1991–2020 climatology)", loc="left")
    axes[-1].set_xlabel("year")
    plt.tight_layout(); plt.savefig(f"{FIGS}/fwi_projections.png", dpi=150); plt.show()
else:
    print("No projection runs available yet — run section 4 once the CMIP6 downloads finish.")

### 5.1 Scenario separation

Over 2026–2045 the forced response differences between SSP pathways remain small; scenarios do not
separate cleanly until around mid-century, and much of the gap sits inside internal variability.
The two lines will look similar.

**That similarity is a finding, not a failure.** Near-term fire-weather risk is largely already
committed regardless of emissions path — which is precisely the message an adaptation and
emergency-planning audience needs.

In [ ]:
if len(annuals) >= 2:
    rows = []
    for city in SITES:
        a = annuals.get((city, "ssp245"))
        b = annuals.get((city, "ssp585"))
        if a is None or b is None:
            continue
        for metric in ["days_fwi_gt30", "longest_high_run"]:
            rows.append({
                "city": SITES[city]["label"],
                "metric": metric,
                "SSP2-4.5": round(a[metric].mean(), 1),
                "SSP5-8.5": round(b[metric].mean(), 1),
                "difference": round(b[metric].mean() - a[metric].mean(), 1),
                "interannual_sd": round(
                    pd.concat([a[metric], b[metric]]).std(), 1),
            })
    sep = pd.DataFrame(rows)
    display(sep)
    print("\nWhere 'difference' is smaller than 'interannual_sd', the scenarios are")
    print("not separable at this horizon — state that explicitly rather than")
    print("presenting the gap as a meaningful projection.")
else:
    print("Need both scenarios for this comparison.")

## 6. Caveats to carry onto the slide

Fill the bracketed values from the outputs above before presenting.

1. **Wind held fixed.** Wind held at the ERA5 1991–2020 monthly climatology; projected FWI change
   reflects temperature, humidity and precipitation only. The climatology is built from daily-mean
   wind, which runs below noon wind, so projected FWI is conservative rather than inflated.

2. **Validated, not calibrated.** FWI computed from daily fields rather than noon observations;
   validated against CEMS over 1991–2025 with r = **[§2.1]** and a mean offset of **[§2.1]** FWI
   units. Exceedance frequency and persistence are the reported quantities; absolute FWI magnitude
   is not.

3. **Single model, no bias correction.** One CMIP6 model (EC-Earth3-CC), ERA5-derived 1990–2020
   thresholds applied without refitting. Directional risk, not calibrated magnitude. Scenario
   spread is not an uncertainty range — it is one model's response under two forcing pathways.

4. **Near-term horizon.** 2026–2045 is too early for SSP pathways to separate cleanly. Similarity
   between scenarios reflects committed warming, not a modelling artefact.

5. **RH route differs between arms.** The historical validation uses ERA5 `rh_min` directly; the
   projection derives RH from specific humidity, sea-level pressure and daily maximum temperature.
   Both target noon humidity, so the validation tests the FWI implementation but not the RH
   derivation.

---

### Open items

- [ ] Confirm whether Larissa has CEMS FWI coverage; the fire arm may be Seville-only.
- [ ] Check whether EWDS offers an uncorrected reanalysis FWI variant — validating against that
      instead would remove the bias-correction confound from §2.1.
- [ ] Optional: pull EC-Earth3 extreme indices from `sis-extreme-indices-cmip6` as second-model
      spread. Not a cross-check (different model), but the only ensemble spread the study could show.